In [ ]:
# ==============================================================================
# CELL 1: Environment Setup & Secure File Download (Header Auth Version)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
import urllib.request
import os
import getpass
import gzip

print("Project Hee-Haw: ML Pipeline Initialized.")

# 1. Securely request the GitHub PAT
print("Please enter your GitHub Personal Access Token (Input will be hidden):")
# We add .strip() to automatically remove any accidental spaces you might paste!
github_token = getpass.getpass().strip()

# 2. Standard URL (No token injected here!)
repo_path = "AdvaittejPS/HeeHaw/main/activity_logs/infected_05_fsm.vcd.gz"
raw_url = f"https://raw.githubusercontent.com/{repo_path}"
vcd_filename = "infected_05_fsm.vcd.gz"

# 3. Download using a secure HTTP Header
print(f"\n[*] Fetching '{vcd_filename}' securely from private GitHub repo...")
try:
    # Build the request and attach the ID badge invisibly
    req = urllib.request.Request(raw_url)
    req.add_header('Authorization', f'token {github_token}')
    req.add_header('Accept', 'application/vnd.github.v3.raw')

    # Execute the request and save the file
    with urllib.request.urlopen(req) as response, open(vcd_filename, 'wb') as out_file:
        out_file.write(response.read())

    print(f"[+] Successfully downloaded {vcd_filename} ({os.path.getsize(vcd_filename) / 1024 / 1024:.2f} MB)")
except Exception as e:
    print(f"[!] Error downloading file. Details: {e}")

In [ ]:
# ==============================================================================
# CELL 2: 2D VCD Data Extractor
# Parses the massive VCD file to extract both Total Switches & Timestamp of Last Switch for every single wire.
# ==============================================================================
def parse_vcd_2d(file_path):
    print(f"[*] Parsing VCD file: {file_path}...")
    signals = {}       # Map of ID -> Name
    switches = {}      # Map of ID -> Switch Count
    last_time = {}     # Map of ID -> Timestamp of Last Switch

    current_scope = []
    current_time = 0

    # NEW: Smart open function detects if the file is zipped
    open_func = gzip.open if file_path.endswith('.gz') else open

    # NEW: Use 'rt' (Read Text) so the zipped bytes are converted back to strings
    with open_func(file_path, 'rt') as f:
        for line in f:
            line = line.strip()
            if not line: continue

            # --- HEADER: Build the Hierarchy ---
            if line.startswith('$scope'):
                parts = line.split()
                if len(parts) >= 3: current_scope.append(parts[2])
            elif line.startswith('$upscope'):
                if current_scope: current_scope.pop()

            # --- HEADER: Map the Signals ---
            elif line.startswith('$var'):
                parts = line.split()
                if len(parts) >= 5:
                    sig_id = parts[3]
                    sig_name = "/".join(current_scope) + "/" + parts[4]
                    signals[sig_id] = sig_name
                    switches[sig_id] = 0
                    last_time[sig_id] = 0

            # --- DATA: Track Time ---
            elif line.startswith('#'):
                current_time = int(line[1:])

            # --- DATA: Track Switching Activity ---
            elif not line.startswith('$'):
                if line.startswith('b') or line.startswith('B'):
                    # Multi-bit vector
                    parts = line.split()
                    if len(parts) == 2:
                        sig_id = parts[1]
                        if sig_id in switches:
                            switches[sig_id] += 1
                            last_time[sig_id] = current_time
                else:
                    # Single-bit scalar
                    sig_id = line[1:].strip()
                    if sig_id in switches:
                        switches[sig_id] += 1
                        last_time[sig_id] = current_time

    print(f"[+] Parsing complete. Extracted {len(signals)} total signals.")
    return signals, switches, last_time

# Execute the parser
if os.path.exists(vcd_filename):
    sigs, sw_counts, times = parse_vcd_2d(vcd_filename)
else:
    print("[!] VCD file not found. Please run Cell 1 successfully first.")

In [ ]:
# ==============================================================================
# CELL 3: Data Structuring & Filtering
# Convert the raw dictionaries into a Pandas DataFrame and filter out the clock.
# ==============================================================================
if 'sigs' in locals():
    data = []
    for sig_id, name in sigs.items():
        # We manually filter 'clk' and 'reset' because they dominate the scale and
        # compress the visual graph.
        # Notice we DO NOT filter 'rcon' anymore! The ML must figure that out itself.
        if "clk" in name.lower() or "rst" in name.lower() or "reset" in name.lower():
            continue

        data.append({
            'ID': sig_id,
            'Name': name,
            'Switches': sw_counts[sig_id],
            'Last_Switch_Time': times[sig_id]
        })

    df = pd.DataFrame(data)
    print(f"[*] Dataset prepared for ML: {len(df)} logic gates ready for analysis.")
    display(df.head())

In [ ]:
# ==============================================================================
# CELL 4: Unsupervised Machine Learning (Isolation Forest)
# Feed the 2D data to the AI. It will isolate the mathematical outliers.
# ==============================================================================
if 'df' in locals():
    print("[*] Training Isolation Forest Model...")

    # INCREASED contamination to 2% to bypass the static-gate tie-breaker quirk!
    model = IsolationForest(contamination=0.02, random_state=42)

    # Features: X-axis (Total Switches) and Y-axis (Time of Last Switch)
    features = df[['Switches', 'Last_Switch_Time']]

    # The model outputs '1' for Normal behavior, and '-1' for Anomalies!
    df['Anomaly_Label'] = model.fit_predict(features)

    # Extract only the anomalies
    trojans_detected = df[df['Anomaly_Label'] == -1].sort_values(by='Switches')

    print("\n========================================================")
    print("              🚨 AI DETECTION REPORT 🚨")
    print("========================================================")
    # Display the top anomalies
    display(trojans_detected.head(15))

In [ ]:
# ==============================================================================
# CELL 5: Data Visualization (Upgraded with Smart Labels)
# ==============================================================================
# 1. Install the adjustText library silently in Colab
!pip install adjustText -q

from adjustText import adjust_text

if 'df' in locals():
    plt.figure(figsize=(14, 8))
    sns.set_theme(style="darkgrid")

    # Plot Normal Logic (Blue)
    normal_data = df[df['Anomaly_Label'] == 1]
    plt.scatter(normal_data['Switches'], normal_data['Last_Switch_Time'],
                color='royalblue', alpha=0.4, s=30, label='Normal AES Logic')

    # Plot Anomalies / Trojans (Red X)
    anomaly_data = df[df['Anomaly_Label'] == -1]
    plt.scatter(anomaly_data['Switches'], anomaly_data['Last_Switch_Time'],
                color='red', marker='X', s=150, edgecolor='black', linewidth=1,
                label='Detected Anomalies (Trojans / Constants)')

    plt.title("Project Hee-Haw: Isolation Forest Hardware Trojan Detection", fontsize=18, fontweight='bold', pad=20)
    plt.xlabel("Total Switching Activity (Entropy)", fontsize=14)
    plt.ylabel("Timestamp of Last Switch (ns)", fontsize=14)
    plt.legend(fontsize=12, loc='lower right')

    # 2. Smart Labeling Strategy
    texts = []
    # We can safely label more points now (e.g., top 8) because they won't overlap!
    for i in range(min(10, len(anomaly_data))):
        row = anomaly_data.iloc[i]
        short_name = row['Name'].split('/')[-1] # Grab just the wire name
        # Instead of plotting immediately, we collect the text objects
        texts.append(plt.text(row['Switches'], row['Last_Switch_Time'], short_name,
                              fontsize=11, color='darkred', fontweight='bold'))

    # 3. The Magic: Automatically repel labels and draw connecting arrows
    adjust_text(texts, arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

    plt.tight_layout()
    plt.show()